# 제출 코드 실행

최종 test 마스킹은 **09번만 실행**합니다. 01~08번은 학습 코드입니다.

## 입력

- test.zip 또는 test 이미지 폴더.
- runs/yolo_eu_03_sizebalanced_20261011_seed2026/best50.pt.
- 다음 5개 run의 best.pt, export_index.json, 연결된 exports/.../manifest.json.

| run | 모델 |
|---|---|
| dinov3_hard_ablation_control_01 | DINO |
| specialist_primary_fast_01 | Primary Si |
| specialist_al3ni_fast_01 | Al3Ni |
| advanced_eu_balanced_ram_01 | 개선 Eu |
| specialist_al_fast_01 | Al |

**기존 valid/test .npz 예측 확률 파일은 사용하지 않습니다.** JSON metadata는 기존 선택 TTA와 모델 SHA 확인에 필요합니다. 9번 RUN_NAMES에 6개 run 폴더를 지정합니다. 과거 모델의 고정 SHA·epoch·점수와 비교하지 않습니다. 선택한 가중치와 같은 run의 export가 일치하는지 확인하며 SHA는 실행 기록에도 남깁니다. CHECKPOINT_NAMES에서 가중치 파일명도 지정할 수 있습니다.

## 실행

코드 폴더를 MyDrive/KoMaP2026_Submission_DirectInference_20261015에 두고, 09번의 CODE_ROOT·PROJECT_ROOT·TEST_INPUT·OUTPUT_DIR·RUN_NAMES를 지정합니다. test.zip은 파일을 디스크에 풀지 않고 이미지만 읽습니다. images 폴더를 우선하며 masks/labels 폴더와 _mask.png는 제외합니다. 이미지 개수 제한 없이 입력 이미지를 처리합니다.

출력: OUTPUT_DIR/masks/*.png, test_masks.zip, inference_report.json. 마스크는 입력의 원본 크기입니다. _image.png는 _mask.png로 바꿉니다. 클래스는 0 Primary Si, 1 Al3Ni, 2 Eutectic Si, 3 Al이며 원래 RGB 팔레트를 사용합니다. 실행 기록에 사용한 모델 SHA·타일·TTA·GPU를 남깁니다.

일반 실행 예:

`python komap_direct_infer.py --input test.zip --output masks_result --runs /path/to/KoMaP2026/runs`

가중치/metadata만 검사: `python komap_direct_infer.py --runs /path/to/runs --check-assets`

## 환경과 검증

environment.yaml은 첨부 환경에서 프로젝트 관련 패키지만 고른 설치 명세입니다. timm 1.0.30, OpenCV 일반 패키지 4.14.0.94 하나로 맞췄습니다. Colab 내부 파일 경로와 python-apt 등 무관한 시스템 패키지는 제외했습니다. 새 환경 설치의 전체 검증은 수행하지 않았습니다. 이미 동작하는 코랩에서는 09번이 설치하지 않고 확인만 합니다.

YOLO는 저장된 실행 backend를 사용합니다. DINO·개선 Eu의 AMP는 원래 export 함수처럼 GPU의 bf16 지원 여부에 따라 선택합니다. 장치와 연산 정밀도가 바뀌면 확률이 조금 달라질 수 있습니다. 결합 비율과 후처리를 유지합니다. 새 가중치를 선택하면 결과도 달라집니다. 실제 가중치·CUDA 전체 실행과 0.8001 재현은 로컬에서 검증하지 못했습니다.

원본 가중치·데이터는 코드 ZIP에 포함하지 않습니다. 01~08번의 기존 run 재개/재export에는 원본 코드 SHA가 필요하며, 전체 학습에는 추가 초기 checkpoint가 필요합니다.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/KoMaP2026')
CODE_ROOT = Path('/content/drive/MyDrive/KoMaP2026_Submission_DirectInference_20261015')
TEST_INPUT = PROJECT_ROOT / 'test.zip'
OUTPUT_DIR = PROJECT_ROOT / 'submission_test_masks_01'
RUNS = PROJECT_ROOT / 'runs'
EXPECTED_COUNT = None
# runs 아래 폴더명: 재학습 모델을 쓰려면 여기서 변경합니다.
RUN_NAMES = {
    'yolo': 'yolo_eu_03_sizebalanced_20261011_seed2026',
    'dino': 'dinov3_hard_ablation_control_01',
    'primary': 'specialist_primary_fast_01',
    'ni': 'specialist_al3ni_fast_01',
    'eu': 'advanced_eu_balanced_ram_01',
    'al': 'specialist_al_fast_01',
}
CHECKPOINT_NAMES = {
    'yolo': 'best50.pt',
    'dino': 'best.pt', 'primary': 'best.pt',
    'ni': 'best.pt', 'eu': 'best.pt', 'al': 'best.pt',
}
